# Document Q&A Assistant

In [9]:
import io
from typing import Iterable, Callable
import zipfile
import traceback
from dataclasses import dataclass

import requests

import frontmatter
from typing import Any, Dict, Iterable, List
from minsearch import Index
import json

from openai import OpenAI

### Data preprocessing

In [10]:
@dataclass
class RawRepositoryFile:
    filename: str
    content: str


class GithubRepositoryDataReader:
    """
    Downloads and parses markdown and code files from a GitHub repository.
    """

    def __init__(self,
                repo_owner: str,
                repo_name: str,
                allowed_extensions: Iterable[str] | None = None,
                filename_filter: Callable[[str], bool] | None = None
        ):
        """
        Initialize the GitHub repository data reader.
        
        Args:
            repo_owner: The owner/organization of the GitHub repository
            repo_name: The name of the GitHub repository
            allowed_extensions: Optional set of file extensions to include
                    (e.g., {"md", "py"}). If not provided, all file types are included
            filename_filter: Optional callable to filter files by their path
        """
        prefix = "https://codeload.github.com"
        self.url = (
            f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"
        )

        if allowed_extensions is not None:
            self.allowed_extensions = {ext.lower() for ext in allowed_extensions}

        if filename_filter is None:
            self.filename_filter = lambda filepath: True
        else:
            self.filename_filter = filename_filter

    def read(self) -> list[RawRepositoryFile]:
        """
        Download and extract files from the GitHub repository.
        
        Returns:
            List of RawRepositoryFile objects for each processed file
            
        Raises:
            Exception: If the repository download fails
        """
        resp = requests.get(self.url)
        if resp.status_code != 200:
            raise Exception(f"Failed to download repository: {resp.status_code}")

        zf = zipfile.ZipFile(io.BytesIO(resp.content))
        repository_data = self._extract_files(zf)
        zf.close()

        return repository_data

    def _extract_files(self, zf: zipfile.ZipFile) -> list[RawRepositoryFile]:
        """
        Extract and process files from the zip archive.
        
        Args:
            zf: ZipFile object containing the repository data

        Returns:
            List of RawRepositoryFile objects for each processed file
        """
        data = []

        for file_info in zf.infolist():
            filepath = self._normalize_filepath(file_info.filename)

            if self._should_skip_file(filepath):
                continue

            try:
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode("utf-8", errors="ignore")
                    if content is not None:
                        content = content.strip()

                    file = RawRepositoryFile(
                        filename=filepath,
                        content=content
                    )
                    data.append(file)

            except Exception as e:
                print(f"Error processing {file_info.filename}: {e}")
                traceback.print_exc()
                continue

        return data

    def _should_skip_file(self, filepath: str) -> bool:
        """
        Determine whether a file should be skipped during processing.
        
        Args:
            filepath: The file path to check
            
        Returns:
            True if the file should be skipped, False otherwise
        """
        filepath = filepath.lower()

        # directory
        if filepath.endswith("/"):
            return True

        # hidden file
        filename = filepath.split("/")[-1]
        if filename.startswith("."):
            return True

        if self.allowed_extensions:
            ext = self._get_extension(filepath)
            if ext not in self.allowed_extensions:
                return True

        if not self.filename_filter(filepath):
            return True

        return False

    def _get_extension(self, filepath: str) -> str:
        """
        Extract the file extension from a filepath.
        
        Args:
            filepath: The file path to extract extension from
            
        Returns:
            The file extension (without dot) or empty string if no extension
        """
        filename = filepath.lower().split("/")[-1]
        if "." in filename:
            return filename.rsplit(".", maxsplit=1)[-1]
        else:
            return ""

    def _normalize_filepath(self, filepath: str) -> str:
        """
        Removes the top-level directory from the file path inside the zip archive.
        'repo-main/path/to/file.py' -> 'path/to/file.py'
        
        Args:
            filepath: The original filepath from the zip archive
            
        Returns:
            The normalized filepath with top-level directory removed
        """
        parts = filepath.split("/", maxsplit=1)
        if len(parts) > 1:
            return parts[1]
        else:
            return parts[0]

In [11]:
def read_github_data():
    allowed_extensions = {"md", "mdx"}

    repo_owner = 'evidentlyai'
    repo_name = 'docs'

    reader = GithubRepositoryDataReader(
        repo_owner,
        repo_name,
        allowed_extensions=allowed_extensions,
    )
    
    return reader.read()

In [12]:
github_data = read_github_data()

In [13]:
github_data[40]

RawRepositoryFile(filename='examples/GitHub_actions.mdx', content='---\ntitle: "Evidently and GitHub actions"\ndescription: "Testing LLM outputs as part of the CI/CD flow."\n---\n\nYou can use Evidently together with GitHub Actions to automatically test the outputs of your LLM agent or application - as part of every code push or pull request.\n\n## How the integration work:\n\n- You define a test dataset of inputs (e.g. test prompts with or without reference answers). You can store it as a file, or save the dataset at Evidently Cloud callable by Dataset ID.\n- Run your LLM system or agent against those inputs inside CI.\n- Evidently automatically evaluates the outputs using the user-specified config (which defines the Evidently descriptors, tests and Report composition), including methods like:\n  - LLM judges (e.g., tone, helpfulness, correctness)\n  - Custom Python functions\n  - Dataset-level metrics like classification quality\n- If any test fails, the CI job fails.\n- You get a de

In [14]:
def parse_data(data_raw):
    """
    Parse a list of raw file objects containing front matter and content.

    This function processes a collection of file-like objects, each expected to have
    `content` and `filename` attributes. It uses the `python-frontmatter` library to
    extract metadata and body content from each file, converts the result into a
    dictionary, and appends the original filename to the parsed data.

    Args:
        data_raw (list): A list of RawRepositoryFile objects where each object has:
            - content (str): The text content of the file, including front matter.
            - filename (str): The name of the file.

    Returns:
        list[dict]: A list of dictionaries, each representing the parsed file data,
        including its front matter, body, and filename.
    """
    data_parsed = []
    for f in data_raw:
        post = frontmatter.loads(f.content) # format file content
        data = post.to_dict()
        data['filename'] = f.filename # add back the filename
        data_parsed.append(data)

    return data_parsed

In [15]:
parsed_data = parse_data(github_data)

In [16]:
parsed_data[40]

{'title': 'Evidently and GitHub actions',
 'description': 'Testing LLM outputs as part of the CI/CD flow.',
 'content': 'You can use Evidently together with GitHub Actions to automatically test the outputs of your LLM agent or application - as part of every code push or pull request.\n\n## How the integration work:\n\n- You define a test dataset of inputs (e.g. test prompts with or without reference answers). You can store it as a file, or save the dataset at Evidently Cloud callable by Dataset ID.\n- Run your LLM system or agent against those inputs inside CI.\n- Evidently automatically evaluates the outputs using the user-specified config (which defines the Evidently descriptors, tests and Report composition), including methods like:\n  - LLM judges (e.g., tone, helpfulness, correctness)\n  - Custom Python functions\n  - Dataset-level metrics like classification quality\n- If any test fails, the CI job fails.\n- You get a detailed test report with pass/fail status and metrics.\n\n![]

### Chunking

In [17]:
def sliding_window(
        seq: Iterable[Any],
        size: int,
        step: int
    ) -> List[Dict[str, Any]]:
    """
    Create overlapping chunks from a sequence using a sliding window approach.

    Args:
        seq: The input sequence (string or list) to be chunked.
        size (int): The size of each chunk/window.
        step (int): The step size between consecutive windows.

    Returns:
        list: A list of dictionaries, each containing:
            - 'start': The starting position of the chunk in the original sequence
            - 'content': The chunk content

    Raises:
        ValueError: If size or step are not positive integers.

    Example:
        >>> sliding_window("hello world", size=5, step=3)
        [{'start': 0, 'content': 'hello'}, {'start': 3, 'content': 'lo wo'}]
    """
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step): # sliding window
        batch = seq[i:i+size] # get the next fixed-length sequence of characters
        result.append({'start': i, 'content': batch})
        if i + size > n:
            break

    return result


def chunk_documents(
        documents: Iterable[Dict[str, str]],
        size: int = 2000,
        step: int = 1000,
        content_field_name: str = 'content'
) -> List[Dict[str, str]]:
    """
    Split a collection of documents into smaller chunks using sliding windows.

    Takes documents and breaks their content into overlapping chunks while preserving
    all other document metadata (filename, etc.) in each chunk.

    Args:
        documents: An iterable of document dictionaries. Each document must have a content field.
        size (int, optional): The maximum size of each chunk. Defaults to 2000.
        step (int, optional): The step size between chunks. Defaults to 1000.
        content_field_name (str, optional): The name of the field containing document content.
                                          Defaults to 'content'.

    Returns:
        list: A list of chunk dictionaries. Each chunk contains:
            - All original document fields except the content field
            - 'start': Starting position of the chunk in original content
            - 'content': The chunk content

    Example:
        >>> documents = [{'content': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, size=100, step=50)
        >>> # Or with custom content field:
        >>> documents = [{'text': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, content_field_name='text')
    """
    results = []

    for doc in documents: # loop over structured document dictionary
        doc_copy = doc.copy()
        doc_content = doc_copy.pop(content_field_name) # extract the content field value
        chunks = sliding_window(doc_content, size=size, step=step) # split the content into overlapping chunks of equal length
        for chunk in chunks: # loop over chunks
            chunk.update(doc_copy) # add back the non-content field to each chunk
        results.extend(chunks)

    return results

In [18]:
chunks = chunk_documents(parsed_data)

In [19]:
chunks[4]

{'start': 3000,
 'content': ' label="2025-04-10" description="Evidently v7.0">\n  ## **Evidently 0.7**\n\nThis release introduces breaking changes. Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.0).\n* The new Evidently API becomes the default. Read the [Migration guide](/faq/migration).\n* New Evidently Cloud version released. Read the [Evidently Cloud v2 notice](/faq/cloud_v2).\n</Update>\n\n<Update label="2025-03-31" description="Evidently v0.6.7">\n  ## **Evidently 0.6.7**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.6.7).\n</Update>\n\n<Update label="2025-03-12" description="Evidently v0.6.6">\n  ## **Evidently 0.6.6**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.6.6).\n</Update>\n\n<Update label="2025-02-17" description="Evidently v0.6.5">\n  ## **Evidently 0.6.5**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/re

In [20]:
index = Index(text_fields=["content", "filename", "title", "description"])
index.fit(chunks)

### RAG flow

In [21]:
def search(query):
    return index.search(
        query=query,
        num_results=15
    )

In [22]:
instructions = """
    You're an assistant that helps with the documentation.
    Answer the QUESTION based on the CONTEXT from the search engine of our documentation.

    Use only the facts from the CONTEXT when answering the QUESTION.

    When answering the question, provide the reference to the file with the source.
    Use the filename field for that. The repo url is: https://github.com/evidentlyai/docs/
    Include code examples when relevant. 
    If the question is discussed in multiple documents, cite all of them.

    Don't use markdown or any formatting in the output.
""".strip()

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results)

    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    
    return prompt

In [23]:
openai_client = OpenAI()

def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [24]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    response = llm(prompt)
    return response

In [25]:
question = 'how do I use llm-as-a-judge for evals'

In [26]:
result = rag(question)
print(result)

Using LLM as a judge for evaluations involves setting up a structured evaluation system where a language model assesses responses against predefined criteria. Here's a step-by-step guide based on the context you provided:

### Step-by-Step Guide

1. **Install Required Libraries**:
   Make sure you have the necessary Python library, `evidently`, installed.

   ```bash
   pip install evidently
   ```

2. **Import Required Modules**:
   Import the necessary libraries as shown below:

   ```python
   import pandas as pd
   import numpy as np
   from evidently import Dataset, DataDefinition, Report, BinaryClassification
   from evidently.llm.templates import BinaryClassificationPromptTemplate
   ```

3. **Set Up the OpenAI API Key**:
   Add your OpenAI API key as an environment variable:

   ```python
   import os
   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
   ```

4. **Create Your Evaluation Dataset**:
   You need a dataset that consists of:
   - **Questions** (inputs to the LLM)
   - **T